# 2 — Intermittency

Companion to **section 3**. Same model as notebook 1, now run over many hours
with an availability profile $\gamma_{g,t}$ on wind and solar.

The economics that appears: the price becomes a *distribution*, and a
technology's value comes to depend on **when** it produces.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

In [ ]:
from model import dispatch, dispatch_t
from pipeline.run_dispatch import CAPACITY_MW

tech = dispatch.read_tech(PROCESSED / "technology_costs_small.csv")
profiles = pd.read_csv(PROCESSED / "profiles_dk1_2024.csv",
                       index_col="time", parse_dates=True)

# Every 8th hour of the year keeps this instant and still sees all seasons.
sample = profiles.iloc[::8]
load = sample["load_mw"].rename("load")
availability = pd.DataFrame({
    "solar_pv":      sample["solar_pu"],
    "wind_onshore":  sample["wind_onshore_pu"],
    "wind_offshore": sample["wind_offshore_pu"],
})
print(f"{len(load)} hours of DK1 2024")

In [ ]:
sol = dispatch_t.solve(tech, CAPACITY_MW, load, availability)
price = sol["price"]

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(range(len(price)), sorted(price, reverse=True), color="#0072B2", lw=1.6)
ax.set_xlabel("hours (sorted)")
ax.set_ylabel("price (EUR/MWh)")
ax.set_title("price duration curve", loc="left")
plt.show()

print("mean price", round(price.mean(), 2), "EUR/MWh")

The curve is a **staircase on the fleet's marginal costs** — because in every
hour some technology is marginal, and the price equals its cost.

## Capacity factor, capture price, value factor

Three statistics, all in `model/dispatch_t.py`:

$$\mathit{CF}_g=\frac{\sum_t q_{g,t}}{T\bar q_g},\qquad
  \bar p_g=\frac{\sum_t \lambda_t q_{g,t}}{\sum_t q_{g,t}},\qquad
  \mathit{VF}_g=\frac{\bar p_g}{\bar p},\qquad
  \bar p=\frac{\sum_t \lambda_t D_t}{\sum_t D_t}$$

The value factor's base price $\bar p$ is weighted by *consumption*, not by hours: it is what the average MWh actually costs. A plain time-average would count a cheap surplus hour the same as an expensive peak hour.

In [ ]:
gen = sol["generation"]
pd.DataFrame({
    "capacity factor": dispatch_t.capacity_factor(gen, CAPACITY_MW).round(3),
    "capture price":   dispatch_t.capture_price(price, gen).round(1),
    "value factor":    dispatch_t.value_factor(price, gen, load).round(3),
}).dropna().sort_values("value factor")

## Cannibalisation

Scale wind up and watch its own value collapse. Each point is a full re-solve.

In [ ]:
WIND = ["wind_onshore", "wind_offshore"]
rows = []
for scale in [0.5, 1, 2, 3, 4, 6]:
    cap = CAPACITY_MW.copy()
    cap[WIND] *= scale
    s = dispatch_t.solve(tech, cap, load, availability)
    g, p = s["generation"], s["price"]
    w = g[WIND].sum(axis=1)
    rows.append({"scale": scale,
                 "wind share": w.sum() / load.sum(),
                 "capture price": (p * w).sum() / w.sum(),
                 "avg price": p.mean()})
sweep = pd.DataFrame(rows)
sweep["value factor"] = sweep["capture price"] / sweep["avg price"]
sweep.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(sweep["wind share"] * 100, sweep["capture price"], "o-",
        color="#0072B2", label="wind capture price")
ax.plot(sweep["wind share"] * 100, sweep["avg price"], "--",
        color="#999999", label="average price")
ax.set_xlabel("wind's share of energy (%)")
ax.set_ylabel("EUR/MWh")
ax.legend(frameon=False)
plt.show()

## The ratio trap

Now the subtler experiment (Figure 3.5 of the note). Hold wind **fixed** and
scale *solar*. Watch wind's value factor and wind's revenue move in opposite
directions — because the value factor's denominator is falling faster than
its numerator.

In [ ]:
rows = []
for scale in [1, 2, 4, 8, 16, 24]:
    cap = CAPACITY_MW.copy()
    cap["solar_pv"] *= scale
    s = dispatch_t.solve(tech, cap, load, availability)
    g, p = s["generation"], s["price"]
    w = g[WIND].sum(axis=1)
    capture = (p * w).sum() / w.sum()
    rows.append({"solar scale": scale,
                 "solar share": g["solar_pv"].sum() / load.sum(),
                 "wind value factor": capture / p.mean(),
                 "wind revenue (MEUR)": (p * w).sum() / 1e6})
trap = pd.DataFrame(rows)
trap.round(3)

Read the last two columns against each other. If your run reproduces the
note's, the value factor **rises** while the revenue **falls**.

The lesson generalises: whenever you meet a normalised performance measure,
ask what is in the denominator and whether it is moving.

## Your turn

1. Repeat the cannibalisation sweep for solar instead of wind. Whose value
   falls faster, and why? (Hint: how many hours does each produce in?)
2. Add a carbon price of 85 EUR/t to every solve. Does cannibalisation get
   better or worse?
3. Change `::8` to `::4` in the sampling cell. Which conclusions move?

In [ ]:
# Try it here.